# Embedded Poet

## Set up environment

#### Load corpus

In [2]:
from pathlib import Path
import re
import csv
import pandas as pd

# ---------- Paths (adjust if you renamed) ----------
CSV_IN  = Path("dickinson_prepared_utf8.csv")   # NLP-friendly CSV you created
TXT_IN  = Path("dickinson_prepared.txt")        # optional: for spot checks
STATS_OUT = Path("dickinson_stats.csv")         # output stats CSV


In [3]:
if not CSV_IN.exists():
    raise FileNotFoundError(f"Cannot find {CSV_IN.resolve()}")

df = pd.read_csv(CSV_IN, encoding="utf-8")  # expects columns: id,title,text
required_cols = {"id","title","text"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"CSV must contain columns {required_cols}, found {set(df.columns)}")

# (Optional) quick peek that TXT exists; not used for analysis here
if TXT_IN.exists():
    print(f"Found TXT anthology for reference: {TXT_IN.resolve()}")

Found TXT anthology for reference: C:\Users\dkill\OneDrive\Documents\Poetry\Emily Dickinson\dickinson_prepared.txt


#### NLTK setup

In [10]:
# ---------- NLTK setup (robust) ----------
import nltk

def ensure_nltk_resource(res, path):
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(res)

# Punkt is split in new releases
ensure_nltk_resource("punkt", "tokenizers/punkt")
ensure_nltk_resource("punkt_tab", "tokenizers/punkt_tab")
ensure_nltk_resource("stopwords", "corpora/stopwords")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stops = set(stopwords.words("english"))

# ---------- Tokenizer helper ----------
import re
WORD_RE = re.compile(r"[A-Za-z’']+")  # allow words with apostrophes

def tokenize_words(text: str):
    try:
        toks = [t for t in word_tokenize(text) if WORD_RE.fullmatch(t)]
    except LookupError:
        # emergency fallback: split on whitespace
        toks = re.findall(r"[A-Za-z’']+", text)
    return toks


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dkill\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


#### Helpers

In [11]:
WORD_RE = re.compile(r"[A-Za-z’']+")  # keep words with apostrophes / curly apostrophes
DASH_RE = re.compile(r"[–—-]")        # en/em/regular dash

def tokenize_words(text: str):
    # Use NLTK tokenizer then filter to word-like tokens
    toks = [t for t in word_tokenize(text) if WORD_RE.fullmatch(t)]
    return toks

def end_words(text: str):
    # last word on each non-empty line (lowercased)
    ews = []
    for line in text.splitlines():
        if line.strip():
            ws = WORD_RE.findall(line.lower())
            if ws:
                ews.append(ws[-1])
    return ews

def stanza_count(text: str):
    # Count stanzas as groups separated by >=1 blank line
    # Normalize CRLF to LF first
    blocks = re.split(r"(?:\r?\n){2,}", text.strip())
    return len([b for b in blocks if b.strip()])

def endword_repeat_ratio(ws):
    # simple signal: share of repeated end-words within a poem
    if not ws:
        return 0.0
    return 1.0 - (len(set(ws)) / len(ws))

## Top level stats

In [12]:
stats = []
for _, row in df.iterrows():
    pid   = int(row["id"])
    title = str(row["title"])
    text  = "" if pd.isna(row["text"]) else str(row["text"])

    toks = tokenize_words(text)
    toks_lower = [t.lower() for t in toks]
    toks_ns = [t for t in toks_lower if t not in stops]

    n_tokens = len(toks_lower)
    n_types  = len(set(toks_lower))
    ttr      = (n_types / n_tokens) if n_tokens else 0.0

    n_lines  = len(text.splitlines())
    n_stanzas = stanza_count(text)
    n_chars  = len(text)

    dashes   = len(DASH_RE.findall(text))
    dash_density = dashes / n_tokens if n_tokens else 0.0

    ews = end_words(text)
    ewr = endword_repeat_ratio(ews)

    stats.append({
        "id": pid,
        "title": title,
        "char_count": n_chars,
        "line_count": n_lines,
        "stanza_count": n_stanzas,
        "token_count": n_tokens,
        "type_count": n_types,
        "type_token_ratio": round(ttr, 4),
        "dash_count": dashes,
        "dash_density": round(dash_density, 4),
        "end_words": ";".join(ews),           # semicolon-joined for CSV
        "endword_repeat_ratio": round(ewr, 4),
        "nonstop_token_count": len(toks_ns),  # optional: tokens minus stopwords
    })

stats_df = pd.DataFrame(stats).sort_values("id").reset_index(drop=True)

In [13]:
stats_df.to_csv(
    STATS_OUT,
    index=False,
    encoding="utf-8",
    lineterminator="\n",
    quoting=csv.QUOTE_MINIMAL
)

In [14]:
print("✅ Task 2 complete")
print(f"Read poems: {len(df)}")
print(f"Saved stats: {STATS_OUT.resolve()}")
print("\nCorpus-level quick stats (approx.):")
print(" - Avg tokens/poem:", round(stats_df["token_count"].mean(), 2))
print(" - Median tokens/poem:", int(stats_df["token_count"].median()))
print(" - Avg lines/poem:", round(stats_df["line_count"].mean(), 2))
print(" - Avg dash density:", round(stats_df["dash_density"].mean(), 4))

✅ Task 2 complete
Read poems: 448
Saved stats: C:\Users\dkill\OneDrive\Documents\Poetry\Emily Dickinson\dickinson_stats.csv

Corpus-level quick stats (approx.):
 - Avg tokens/poem: 71.16
 - Median tokens/poem: 51
 - Avg lines/poem: 15.69
 - Avg dash density: 0.0027


## Word-level exploration

In [23]:
import re
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

In [30]:
# --- 0. Stopwords setup ---
STOPS = set(stopwords.words("english"))
CUSTOM_STOPS = {"'s", "'t", '"', "'", "n't", "'m", "'ve", "'ll", "'d"}
STOPS.update(CUSTOM_STOPS)
WORD_RE = re.compile(r"[A-Za-z’']+")

def tokenize_no_stop(text: str):
    """Tokenize text into lowercase words, remove stopwords."""
    toks = [t.lower() for t in word_tokenize(str(text)) if WORD_RE.fullmatch(t)]
    toks = [t for t in toks if t not in STOPS]
    return toks

In [31]:
# --- 1. Tokenize poems into lowercase words ---
docs = [tokenize_no_stop(txt) for txt in df["text"]]

### Unigram frequencies

In [32]:
# --- 2. Unigram frequency table ---
unigram_counts = Counter()
unigram_docfreq = Counter()
for toks in docs:
    unigram_counts.update(toks)
    unigram_docfreq.update(set(toks))

unigram_df = pd.DataFrame([
    {
        "term": term,
        "count": count,
        "doc_freq": unigram_docfreq[term],
        "rel_freq": count / sum(unigram_counts.values())
    }
    for term, count in unigram_counts.most_common()
])
unigram_df.to_csv("unigram_freq.csv", index=False, encoding="utf-8")

### Bigram frequencies

In [37]:
def bigrams(seq):
    return list(zip(seq, seq[1:]))

bigram_counts = Counter()
bigram_docfreq = Counter()
for toks in docs:
    bgs = bigrams(toks)
    bigram_counts.update(bgs)
    bigram_docfreq.update(set(bgs))

bigram_df = pd.DataFrame([
    {
        "bigram": f"{w1} {w2}",
        "count": count,
        "doc_freq": bigram_docfreq[(w1, w2)],
        "rel_freq": count / sum(bigram_counts.values())
    }
    for (w1, w2), count in bigram_counts.most_common()
])
bigram_df.to_csv("bigram_freq.csv", index=False, encoding="utf-8")

In [38]:
# --- 4. TF-IDF top terms per poem ---
docs_joined = [" ".join(toks) for toks in docs]
vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"[A-Za-z’']+",
    stop_words="english",
    min_df=2
)
X = vectorizer.fit_transform(docs_joined)
terms = vectorizer.get_feature_names_out()


In [35]:
TOP_K = 15
tfidf_rows = []
for i, (pid, title) in enumerate(zip(df["id"], df["title"])):
    row = X.getrow(i)
    if row.nnz == 0:
        tfidf_rows.append({"id": pid, "title": title, "top_terms": ""})
        continue
    idx = row.indices
    vals = row.data
    order = vals.argsort()[::-1][:TOP_K]
    top_terms = [f"{terms[idx[j]]}:{vals[j]:.3f}" for j in order]
    tfidf_rows.append({
        "id": pid,
        "title": title,
        "top_terms": ", ".join(top_terms)
    })

In [40]:
tfidf_df = pd.DataFrame(tfidf_rows)
tfidf_df.to_csv("tfidf_top_terms.csv", index=False, encoding="utf-8")

print("✅ Task 3 complete")
print("Unigrams:", len(unigram_df), "rows -> unigram_freq.csv")
print("Bigrams:", len(bigram_df), "rows -> bigram_freq.csv")
print("TF-IDF top terms saved -> tfidf_top_terms.csv")


✅ Task 3 complete
Unigrams: 5279 rows -> unigram_freq.csv
Bigrams: 14565 rows -> bigram_freq.csv
TF-IDF top terms saved -> tfidf_top_terms.csv
